# Baseline — No Augmentation

This notebook will implement the team's agreed no-augmentation baseline. Run `00_dataset_audit.ipynb` first. The classifier, preprocessing, train fractions, and seeds must match the conditions used by the augmentation experiments.

## Planned baseline protocol

- Input: leakage-controlled ASTD training split.
- No augmentation of any kind.
- Train using the agreed low-resource fractions (provisionally 10%, 25%, 50%).
- Fit preprocessing/vectorization on training data only.
- Tune on validation only.
- Evaluate once on the untouched test set.
- Report Accuracy, Macro-Precision, Macro-Recall, Macro-F1, Weighted-F1, and a multiclass confusion matrix.

**Do not fix the classifier yet until the three students agree on the shared classifier.** A classical option is TF-IDF + Linear SVM or Logistic Regression; a transformer option is an Arabic pretrained encoder. The same classification architecture must be used across augmentation conditions for a fair direct comparison.

## 1. Environment Setup, Data Loading & Model Initialization

In [ ]:
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoConfig,
)

# 1. Reproducibility & Project Paths
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "reports" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Load Cleaned Datasets
train_df = pd.read_csv(DATA_DIR / "astd_train.csv")
val_df = pd.read_csv(DATA_DIR / "astd_validation.csv")
test_df = pd.read_csv(DATA_DIR / "astd_test.csv")

print("Shapes -> Train:", train_df.shape, "| Val:", val_df.shape, "| Test:", test_df.shape)
print("\nTrain label distribution:\n", train_df["label"].value_counts())

# 3. Label Mapping & Verification
LABEL2ID = {"OBJ": 0, "NEG": 1, "NEUTRAL": 2, "POS": 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

for name, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    unknown = set(df["label"].unique()) - set(LABEL2ID)
    assert len(unknown) == 0, f"Unknown labels in {name}: {unknown}"

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["label_id"] = train_df["label"].map(LABEL2ID).astype(int)
val_df["label_id"] = val_df["label"].map(LABEL2ID).astype(int)
test_df["label_id"] = test_df["label"].map(LABEL2ID).astype(int)

# 4. Load MARBERT Tokenizer & Config
MODEL_NAME = "UBC-NLP/MARBERT"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(MODEL_NAME)

print("\nModel:", MODEL_NAME, "| Device:", device)
print("Vocab size:", tokenizer.vocab_size, "| Hidden size:", config.hidden_size)

# 5. Dataset & Collate Function
class ArabicTweetDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["text"].astype(str).tolist()
        self.labels = dataframe["label_id"].astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        return {
            "text": self.texts[index],
            "label": self.labels[index]
        }

def collate_batch(batch):
    texts = [item["text"] for item in batch]
    labels = [item["label"] for item in batch]

    tokens = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )
    tokens["labels"] = torch.tensor(labels, dtype=torch.long)
    return tokens

train_dataset = ArabicTweetDataset(train_df)
val_dataset = ArabicTweetDataset(val_df)
test_dataset = ArabicTweetDataset(test_df)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0
)

print("Dataset & Loaders ready -> Val batches:", len(val_loader), "| Test samples:", len(test_dataset))

Shapes -> Train: (6005, 3) | Val: (1997, 3) | Test: (1999, 3)

Train label distribution:
 label
OBJ        4012
NEG        1012
NEUTRAL     500
POS         481
Name: count, dtype: int64


config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


Model: UBC-NLP/MARBERT | Device: cuda
Vocab size: 100000 | Hidden size: 768
Dataset & Loaders ready -> Val batches: 999 | Test samples: 1999


## 2. Low-Resource Nested Splits (10%, 25%, 50%) & Class Weights

In [ ]:
TRAIN_FRACTIONS = [0.10, 0.25, 0.50]

# Shuffle each class once with fixed SEED for nested subsets
shuffled_groups = {
    label: group.sample(frac=1, random_state=SEED).reset_index(drop=True)
    for label, group in train_df.groupby("label")
}

nested_subsets = {}
for fraction in TRAIN_FRACTIONS:
    parts = []
    for label, group in shuffled_groups.items():
        n = round(len(group) * fraction)
        parts.append(group.iloc[:n].copy())

    subset = (
        pd.concat(parts, ignore_index=True)
        .sample(frac=1, random_state=SEED)
        .reset_index(drop=True)
    )
    nested_subsets[fraction] = subset

    print(f"\nTraining fraction: {fraction * 100:.0f}% | Samples: {len(subset)}")
    print("Class counts:", subset["label"].value_counts().to_dict())

# Save version 2 splits and compute class weights from training subsets only
SPLIT_DIR_V2 = DATA_DIR / "low_resource_splits_v2"
SPLIT_DIR_V2.mkdir(parents=True, exist_ok=True)

class_weights_by_fraction = {}

for fraction, subset in nested_subsets.items():
    percentage = int(fraction * 100)
    output_path = SPLIT_DIR_V2 / f"train_{percentage}pct.csv"
    subset.to_csv(output_path, index=False, encoding="utf-8-sig")

    counts = (
        subset["label_id"]
        .value_counts()
        .reindex(range(4), fill_value=0)
    )
    assert (counts > 0).all(), f"Missing class in {percentage}% split"

    weights = len(subset) / (4 * counts)
    class_weights_by_fraction[fraction] = torch.tensor(
        weights.values,
        dtype=torch.float32,
        device=device
    )

    print(f"\n{percentage}% Saved -> {output_path}")
    print("Class weights:", {ID2LABEL[i]: round(float(weights.iloc[i]), 3) for i in range(4)})

train_subsets = nested_subsets
train_loaders = {}


Training fraction: 10% | Samples: 600
Class counts: {'OBJ': 401, 'NEG': 101, 'NEUTRAL': 50, 'POS': 48}

Training fraction: 25% | Samples: 1501
Class counts: {'OBJ': 1003, 'NEG': 253, 'NEUTRAL': 125, 'POS': 120}

Training fraction: 50% | Samples: 3002
Class counts: {'OBJ': 2006, 'NEG': 506, 'NEUTRAL': 250, 'POS': 240}

10% Saved -> /content/data/processed/low_resource_splits_v2/train_10pct.csv
Class weights: {'OBJ': 0.374, 'NEG': 1.485, 'NEUTRAL': 3.0, 'POS': 3.125}

25% Saved -> /content/data/processed/low_resource_splits_v2/train_25pct.csv
Class weights: {'OBJ': 0.374, 'NEG': 1.483, 'NEUTRAL': 3.002, 'POS': 3.127}

50% Saved -> /content/data/processed/low_resource_splits_v2/train_50pct.csv
Class weights: {'OBJ': 0.374, 'NEG': 1.483, 'NEUTRAL': 3.002, 'POS': 3.127}


## 3. MARBERT Baseline Training & Evaluation (10%, 25%, 50%)

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

# Hyperparameters (fixed SEED for reproducibility)
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = criterion(logits, labels)

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / max(len(loader), 1)
    acc = accuracy_score(all_labels, all_preds)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )
    _, _, weighted_f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="weighted", zero_division=0
    )
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(LABEL2ID))))

    return {
        "loss": round(float(avg_loss), 4),
        "accuracy": round(float(acc), 4),
        "macro_precision": round(float(macro_p), 4),
        "macro_recall": round(float(macro_r), 4),
        "macro_f1": round(float(macro_f1), 4),
        "weighted_f1": round(float(weighted_f1), 4),
        "confusion_matrix": cm.tolist(),
        "preds": all_preds,
        "labels": all_labels,
    }


all_results = {}

for fraction in TRAIN_FRACTIONS:
    pct = int(fraction * 100)
    set_seed(SEED)

    subset_df = train_subsets[fraction]
    train_dataset_frac = ArabicTweetDataset(subset_df)
    train_loader_frac = DataLoader(
        train_dataset_frac,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_batch,
        num_workers=0,
    )
    train_loaders[fraction] = train_loader_frac

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABEL2ID),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    ).to(device)

    weights = class_weights_by_fraction[fraction]
    criterion = nn.CrossEntropyLoss(weight=weights)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    total_steps = len(train_loader_frac) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * WARMUP_RATIO),
        num_training_steps=total_steps,
    )

    best_val_macro_f1 = -1.0
    best_model_path = MODEL_DIR / f"marbert_baseline_{pct}pct.pt"

    print(f"\n==================== Training MARBERT ({pct}%) ====================")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss = 0.0

        for batch in train_loader_frac:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / max(len(train_loader_frac), 1)
        val_metrics = evaluate_model(model, val_loader, criterion)

        print(
            f"Epoch {epoch}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val Macro-F1: {val_metrics['macro_f1']:.4f} | "
            f"Val Weighted-F1: {val_metrics['weighted_f1']:.4f}"
        )

        # Tune / select checkpoint on validation set only
        if val_metrics["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["macro_f1"]
            torch.save(model.state_dict(), best_model_path)

    # Load best checkpoint and evaluate once on untouched test set
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    test_metrics = evaluate_model(model, test_loader, criterion)

    print(f"\n--- Test Set Evaluation ({pct}% Baseline) ---")
    for k in ["accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1"]:
        print(f"{k}: {test_metrics[k]:.4f}")
    print("Confusion Matrix (rows=true, cols=pred):\n", np.array(test_metrics["confusion_matrix"]))

    # Save predictions and metrics locally under reports/
    preds_df = pd.DataFrame({
        "true_label_id": test_metrics["labels"],
        "true_label": [ID2LABEL[i] for i in test_metrics["labels"]],
        "pred_label_id": test_metrics["preds"],
        "pred_label": [ID2LABEL[i] for i in test_metrics["preds"]],
    })
    preds_df.to_csv(REPORTS_DIR / f"test_predictions_{pct}pct.csv", index=False, encoding="utf-8-sig")

    all_results[f"{pct}pct"] = {
        "best_val_macro_f1": round(float(best_val_macro_f1), 4),
        "test_metrics": {k: v for k, v in test_metrics.items() if k not in ["preds", "labels"]},
    }

metrics_path = REPORTS_DIR / "baseline_metrics_summary.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

summary_table = pd.DataFrame([
    {
        "Fraction": split,
        "Best Val Macro-F1": res["best_val_macro_f1"],
        "Test Accuracy": res["test_metrics"]["accuracy"],
        "Test Macro-Precision": res["test_metrics"]["macro_precision"],
        "Test Macro-Recall": res["test_metrics"]["macro_recall"],
        "Test Macro-F1": res["test_metrics"]["macro_f1"],
        "Test Weighted-F1": res["test_metrics"]["weighted_f1"],
    }
    for split, res in all_results.items()
])

print("\n==================== Baseline Summary Table ====================")
display(summary_table)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  654MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider


==================== Training MARBERT (10%) ====================


model.safetensors: reconstructing file:   0%|          |  0.00B /  654MB            

... (5 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider


==================== Training MARBERT (50%) ====================
Epoch 1/3 | Train Loss: 1.1822 | Val Loss: 1.0520 | Val Macro-F1: 0.4882 | Val Weighted-F1: 0.6065
Epoch 2/3 | Train Loss: 0.7888 | Val Loss: 0.8286 | Val Macro-F1: 0.5781 | Val Weighted-F1: 0.7147
Epoch 3/3 | Train Loss: 0.5093 | Val Loss: 0.8655 | Val Macro-F1: 0.5647 | Val Weighted-F1: 0.7119

--- Test Set Evaluation (50% Baseline) ---
accuracy: 0.6763
macro_precision: 0.5451
macro_recall: 0.6119
macro_f1: 0.5632
weighted_f1: 0.6992
Confusion Matrix (rows=true, cols=pred):
 [[967 126 167  78]
 [ 60 203  65   8]
 [ 28  38  91   9]
 [ 44   6  18  91]]

==================== Baseline Summary Table ====================


,Fraction,Best Val Macro-F1,Test Accuracy,Test Macro-Precision,Test Macro-Recall,Test Macro-F1,Test Weighted-F1
0,10pct,0.4983,0.6433,0.4643,0.5129,0.4797,0.6576
1,25pct,0.5563,0.6833,0.5296,0.5960,0.5526,0.6965
2,50pct,0.5781,0.6763,0.5451,0.6119,0.5632,0.6992
